In [68]:

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
import seaborn as sns
from matplotlib.image import imread
from PIL import Image
import tensorflow as tf
np.random.seed(1337)
import gc

import glob

from tensorflow.keras import layers
from keras.callbacks import ReduceLROnPlateau
from tensorflow.keras.models import Sequential,Model
from tensorflow.keras.layers import Input, Activation, Dropout, AveragePooling2D,Flatten, Dense, Conv2D,MaxPool2D, MaxPooling2D, BatchNormalization,Conv2DTranspose,concatenate
from tensorflow.keras.callbacks import EarlyStopping
import warnings
warnings.filterwarnings('ignore')


img_size = 128
print(os.listdir())
dataset = os.listdir("music_seperation_dataset/train")
labels = dataset
print(labels)

['.git', '.vscode', 'model_classes_3.ipynb', 'model_classes_full.ipynb', 'model_separation.ipynb', 'music_dataset_spectro_3_instrument', 'music_dataset_spectro_full', 'music_seperation_dataset', 'my_model_3.keras', 'my_model_full.keras', 'README.md', 'spectrogramMaker.py']
['Acoustic_Guitar', 'Bass_Guitar', 'Drum_set', 'Electric_Guitar', 'full_mix', 'Keyboard']


In [88]:
def get_dataset_array(data_dir):
    data = []
    path = os.path.join(data_dir)
    for img in os.listdir(data_dir):
            try:
                img_arr = cv2.imread(os.path.join(path,img))
                resized_arr = cv2.resize(img_arr, (img_size, img_size))
                data.append([resized_arr])
                gc.collect()
            except Exception as e:
                print(e)
    return np.array(data,dtype="float32") 

In [89]:
#train = get_dataset_array("music_seperation_dataset/train/")
x_train = get_dataset_array("music_seperation_dataset/train/full_mix")
y_train = get_dataset_array("music_seperation_dataset/train/Drum_set")

x_test = get_dataset_array("music_seperation_dataset/test/full_mix")
y_test = get_dataset_array("music_seperation_dataset/test/Drum_set")

x_valid = get_dataset_array("music_seperation_dataset/valid/full_mix")
y_valid = get_dataset_array("music_seperation_dataset/valid/Drum_set")
#test = get_dataset_array("music_seperation_dataset/test/")
#valid = get_dataset_array("music_seperation_dataset/valid/")


(48, 2)
(6, 2)
(6, 2)


In [90]:
gc.collect()
x_train = np.array(x_train)/255
gc.collect()
x_test = np.array(x_test)/255
gc.collect()
x_valid = np.array(x_valid)/255
gc.collect()
y_train = np.array(y_train)/255
gc.collect()
y_test = np.array(y_test)/255
gc.collect()
y_valid = np.array(y_valid)/255
gc.collect()

0

In [91]:
x_train = x_train.reshape(-1, img_size, img_size, 3)
y_train = y_train.reshape(-1, img_size, img_size, 3)

x_valid = x_valid.reshape(-1, img_size, img_size, 3)
y_valid = y_valid.reshape(-1, img_size, img_size, 3)

x_test = x_test.reshape(-1, img_size, img_size, 3)
y_test = y_test.reshape(-1, img_size, img_size, 3)

print(x_train.dtype)
print(x_train.shape)

float32
(48, 128, 128, 3)


In [ ]:
num_classes = 3
def Unet():
    inputs =  layers.Input(shape=(128,128,3))

    conv1 = Conv2D(16, (3,3), activation = 'relu', padding='same')(inputs)
    conv1 = Conv2D(16, (3,3), activation = 'relu', padding='same')(conv1)
    pool1 = MaxPool2D((2,2))(conv1)

    conv2 = Conv2D(32, (3,3), activation = 'relu', padding='same')(pool1)
    conv2 = Conv2D(32, (3,3), activation = 'relu', padding='same')(conv2)
    pool2 = MaxPool2D((2,2))(conv2)

    conv3 = Conv2D(64, (3,3), activation = 'relu', padding='same')(pool2)
    conv3 = Conv2D(64, (3,3), activation = 'relu', padding='same')(conv3)
    pool3 = MaxPool2D((2,2))(conv3)

    conv4 = Conv2D(128, (3,3), activation = 'relu', padding='same')(pool3)
    conv4 = Conv2D(128, (3,3), activation = 'relu', padding='same')(conv4)
    pool4 = MaxPool2D((2,2))(conv4)

    
    conv5 = Conv2D(256, (3,3), activation = 'relu', padding='same')(pool4)
    conv5 = Conv2D(256, (3,3), activation = 'relu', padding='same')(conv5)

    goUp1 = Conv2DTranspose(128,(2,2),strides=(2,2),padding='same')(conv5)
    goUp1 = concatenate([goUp1,conv4])
    conv6 = Conv2D(128, (3,3), activation = 'relu', padding='same')(goUp1)
    conv6 = Conv2D(128, (3,3), activation = 'relu', padding='same')(conv6)

    goUp2 = Conv2DTranspose(64,(2,2),strides=(2,2),padding='same')(conv6)
    goUp2 = concatenate([goUp2,conv3])
    conv7 = Conv2D(64, (3,3), activation = 'relu', padding='same')(goUp2)
    conv7 = Conv2D(64, (3,3), activation = 'relu', padding='same')(conv7)

    goUp3 = Conv2DTranspose(32,(2,2),strides=(2,2),padding='same')(conv7)
    goUp3 = concatenate([goUp3,conv2])
    conv8 = Conv2D(32, (3,3), activation = 'relu', padding='same')(goUp3)
    conv8 = Conv2D(32, (3,3), activation = 'relu', padding='same')(conv8)

    goUp4 = Conv2DTranspose(16,(2,2),strides=(2,2),padding='same')(conv8)
    goUp4 = concatenate([goUp4,conv1])
    conv9 = Conv2D(16, (3,3), activation = 'relu', padding='same')(goUp4)
    conv9 = Conv2D(16, (3,3), activation = 'relu', padding='same')(conv9)



    outputs = Conv2D(num_classes,(1,1),activation="sigmoid")(conv9)

    model = Model(inputs=[inputs],outputs=[outputs])
    return model

model = Unet()
model.compile(
              optimizer = 'adam', loss = 'binary_crossentropy',
              metrics = ['accuracy']
              )
     

In [165]:
model.summary()

Model: "functional_17"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_22      │ (None, 128, 128,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_311 (Conv2D) │ (None, 128, 128,  │        448 │ input_layer_22[0… │
│                     │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_312 (Conv2D) │ (None, 128, 128,  │      2,320 │ conv2d_311[0][0]  │
│                     │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_63    │ (None, 64, 64,    │          0 │ conv2d_312[0][0]  │
│ (MaxPooling2D)      │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_313 (Conv2D) │ (None, 64, 64,    │      4,640 │ max_pooling2d_63… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_314 (Conv2D) │ (None, 64, 64,    │      9,248 │ conv2d_313[0][0]  │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_64    │ (None, 32, 32,    │          0 │ conv2d_314[0][0]  │
│ (MaxPooling2D)      │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_315 (Conv2D) │ (None, 32, 32,    │     18,496 │ max_pooling2d_64… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_316 (Conv2D) │ (None, 32, 32,    │     36,928 │ conv2d_315[0][0]  │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_65    │ (None, 16, 16,    │          0 │ conv2d_316[0][0]  │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_317 (Conv2D) │ (None, 16, 16,    │     73,856 │ max_pooling2d_65… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_318 (Conv2D) │ (None, 16, 16,    │    147,584 │ conv2d_317[0][0]  │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_66    │ (None, 8, 8, 128) │          0 │ conv2d_318[0][0]  │
│ (MaxPooling2D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_319 (Conv2D) │ (None, 8, 8, 256) │    295,168 │ max_pooling2d_66… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_320 (Conv2D) │ (None, 8, 8, 256) │    590,080 │ conv2d_319[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_transpose_62 │ (None, 16, 16,    │    131,200 │ conv2d_320[0][0]  │
│ (Conv2DTranspose)   │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_62      │ (None, 16, 16,    │          0 │ conv2d_transpose… │
│ (Concatenate)       │ 256)              │            │ conv2d_318[0][0]  │
├─────────────────────┼───────────────────┼────────────┼─────────────────

 Total params: 1,941,139 (7.40 MB)

 Trainable params: 1,941,139 (7.40 MB)

 Non-trainable params: 0 (0.00 B)

In [166]:
learning_rate_reduction = ReduceLROnPlateau(monitor = 'val_accuracy', patience = 2, verbose = 1, factor = 0.3, min_lr = 0.000001)

In [167]:
batch_size = 32
n_epochs = 10
model.fit(x_train, y_train, batch_size = batch_size,
                    epochs = n_epochs, validation_data =(x_valid, y_valid),
                    callbacks = [learning_rate_reduction])

Epoch 1/10
2/2 ━━━━━━━━━━━━━━━━━━━━ 4s 565ms/step - accuracy: 0.0458 - loss: 0.6902 - val_accuracy: 0.4025 - val_loss: 0.6646 - learning_rate: 0.0010
Epoch 2/10
2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 293ms/step - accuracy: 0.4026 - loss: 0.6626 - val_accuracy: 0.4470 - val_loss: 0.6492 - learning_rate: 0.0010
Epoch 3/10
2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 287ms/step - accuracy: 0.4350 - loss: 0.6485 - val_accuracy: 0.4153 - val_loss: 0.6365 - learning_rate: 0.0010
Epoch 4/10
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 199ms/step - accuracy: 0.4148 - loss: 0.6368
Epoch 4: ReduceLROnPlateau reducing learning rate to 0.0003000000142492354.
2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 285ms/step - accuracy: 0.4146 - loss: 0.6360 - val_accuracy: 0.3774 - val_loss: 0.6234 - learning_rate: 0.0010
Epoch 5/10
2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 286ms/step - accuracy: 0.3880 - loss: 0.6242 - val_accuracy: 0.3902 - val_loss: 0.6181 - learning_rate: 3.0000e-04
Epoch 6/10
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 196ms/step - accuracy: 0.4002 - loss: 0.6195
Epoch 6: Redu

In [168]:
gc.collect()
print("Accuracy of the model is - " , model.evaluate(x_test,y_test)[1]*100 , "%")
model.save('my_model_full.keras')

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 63ms/step - accuracy: 0.4201 - loss: 0.6028
Accuracy of the model is -  42.010498046875 %


In [170]:
img = cv2.imread("music_seperation_dataset/train/full_mix/0_spect.png")
img=cv2.resize(img,(img_size,img_size))
img = np.array(img,"float32")
img= np.reshape(img,(-1, img_size, img_size, 3))
img=img/255
print(img.shape)
predimg =model.predict(img)

predimg.argmax()
predimg2 =Image.fromarray(predimg)



(1, 128, 128, 3)
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step


TypeError: Cannot handle this data type: (1, 1, 128, 3), <f4